1. **Image Preprocessing**
    -   **Dataset Augmentation**
        *   x10 the training dataset
    -   Grayscaling
        *   Primarily reduce the computational power
2. **Intuitively Group Age Classes**
    -   Roblox: 5-8, 9-12, 13-15, 16-17, 18-20, and 21+
    -   Business-wise, we have to add more label for 21+
    -   Suggestion for 21+ group: 21-27, 28-45, 46-65, 66+
        -   21-27 years: Categorized as mature adults (8:23).
        -   28-45 years: Representing the adult stage of life (8:26).
        -   46-65 years: Distinguished by the appearance of more pronounced aging features (8:29).
        -   66+ years: Represented by features like graying hair and aging skin (8:36).
3. Use CNN-Based Deep Learning Aproach


Concluded labels: 1-4, 5-8, 9-12, 13-15, 16-17, 18-20, 21-27, 28-45, 46-65, 66+

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras import mixed_precision
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Flatten, Dense, Dropout, BatchNormalization
# from tensorflow.keras.preprocessing.image import ImageDataGenerator # From old Keras' API. For data augmentation, Keras has imported a new API in TensorFlow 2.x. So, we can use it directly from TensorFlow.
from keras.utils import to_categorical
from keras.optimizers import SGD, Adam
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

# For hashing images and finding duplications:
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import hashlib

import imagehash 
from PIL import Image, ImageOps
from sklearn.model_selection import GroupShuffleSplit

I0000 00:00:1791802164.375155   96420 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791802164.410851   96420 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791802165.568766   96420 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
gpus = tf.config.list_physical_devices("GPU")

print("TensorFlow version:", tf.__version__)
print("GPU devices:", gpus)

TensorFlow version: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# This will reduce the VRAM consumed
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU Memory Growth Enabled!")

GPU Memory Growth Enabled!


In [4]:
# For strong GPUs, execute this code cell to turn on FP16 and reduce VRAM usage consumed
mixed_precision.set_global_policy("mixed_float16")

In [5]:
from pathlib import Path
import kagglehub

# Find the root folder of the project
PROJECT_ROOT = Path.cwd()

# This is for notebooks that is opened in notebooks/ (folder)
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# The function to download dataset immediately into data/raw
def download_dataset(kaggle_url: str, folder_name: str) -> Path:
    """Download a Kaggle dataset into its own directory.
    - kaggle_url: author/dataset-name (on Kaggle)
    - folder_name: the name of the folder that will hold the dataset"""
    output_dir = DATA_DIR / folder_name
    output_dir.mkdir(parents=True, exist_ok=True)

    path = kagglehub.dataset_download(
        kaggle_url,
        output_dir=str(output_dir),
    )

    print(f"Dataset: {kaggle_url}")
    print(f"Saved to: {path}")

    return Path(path)

In [7]:
facial_age_v1_path = download_dataset("frabbisw/facial-age","facial-age-v1")
facial_age_v2_path = download_dataset("frabbisw/facial-age/version/2", "facial-age-v2")
utk_path = download_dataset("jangedoo/utkface-new", "utkface")

100%|██████████| 840M/840M [02:51<00:00, 5.14MB/s] 

Extracting files...


Dataset: frabbisw/facial-age
Saved to: /home/sisi/projects/age-estimation/data/raw/facial-age-v1


100%|██████████| 211M/211M [00:53<00:00, 4.17MB/s] 

Extracting files...


Dataset: frabbisw/facial-age/version/2
Saved to: /home/sisi/projects/age-estimation/data/raw/facial-age-v2


100%|██████████| 331M/331M [01:51<00:00, 3.13MB/s] 

Extracting files...


Dataset: jangedoo/utkface-new
Saved to: /home/sisi/projects/age-estimation/data/raw/utkface
